# SAM2 Pipeline Diagnostics

Investigates why SAM2 produces ~14K detections on the full raster vs YOLO's ~44K.
Two hypotheses:
- **Case 1:** SAM2 mask failures before NMS — pre-NMS count is already ~14K
- **Case 2:** NMS too aggressive on larger bboxes — pre-NMS count is ~44K but NMS drops ~30K

In [ ]:
import sys
sys.path.insert(0, "/home/users/cayleigh/YOLOv8-BeyondEarth/src")
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

sam2_dir = Path.home() / "tmp/YOLOv8BeyondEarth/exp_sam2_256"
yolo_dir = Path.home() / "tmp/YOLOv8BeyondEarth/exp_yolo_256"

## 1. Pre-NMS vs Post-NMS counts

In [ ]:
sam2_pre  = sorted(sam2_dir.glob("*-mask.shp"))
sam2_post = sorted(sam2_dir.glob("*-mask-nms.shp"))
yolo_pre  = sorted(yolo_dir.glob("*-downscaled-mask.shp"))
yolo_post = sorted(yolo_dir.glob("*-downscaled-mask-nms.shp"))

for label, pre, post in [("SAM2", sam2_pre, sam2_post), ("YOLO", yolo_pre, yolo_post)]:
    for a, b in zip(pre, post):
        n_pre  = len(gpd.read_file(a))
        n_post = len(gpd.read_file(b))
        print(f"{label}  pre-NMS: {n_pre:6d}   post-NMS: {n_post:6d}   dropped by NMS: {n_pre - n_post:6d}")

## 2. Score distributions — pre vs post NMS

In [ ]:
gdf_sam2_pre  = gpd.read_file(sam2_pre[0])
gdf_sam2_post = gpd.read_file(sam2_post[0])
gdf_yolo_pre  = gpd.read_file(yolo_pre[0])
gdf_yolo_post = gpd.read_file(yolo_post[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(gdf_sam2_pre["score"],  bins=30, alpha=0.6, label=f"pre-NMS (n={len(gdf_sam2_pre)})")
axes[0].hist(gdf_sam2_post["score"], bins=30, alpha=0.6, label=f"post-NMS (n={len(gdf_sam2_post)})")
axes[0].set_title("SAM2 score distribution"); axes[0].set_xlabel("score"); axes[0].legend()

axes[1].hist(gdf_yolo_pre["score"],  bins=30, alpha=0.6, label=f"pre-NMS (n={len(gdf_yolo_pre)})")
axes[1].hist(gdf_yolo_post["score"], bins=30, alpha=0.6, label=f"post-NMS (n={len(gdf_yolo_post)})")
axes[1].set_title("YOLO score distribution"); axes[1].set_xlabel("score"); axes[1].legend()

plt.tight_layout(); plt.savefig("diag_score_distributions.png", dpi=150); plt.show()

## 3. is_within_slice breakdown

In [ ]:
print("SAM2 pre-NMS — isin_slice breakdown:")
print(gdf_sam2_pre["isin_slice"].value_counts())
print(f"  score=0.10 (edge): {(gdf_sam2_pre['score'] == 0.10).sum()}")
print(f"  score>0.10 (interior): {(gdf_sam2_pre['score'] > 0.10).sum()}")

print("\nYOLO pre-NMS — isin_slice breakdown:")
print(gdf_yolo_pre["isin_slice"].value_counts())
print(f"  score=0.10 (edge): {(gdf_yolo_pre['score'] == 0.10).sum()}")
print(f"  score>0.10 (interior): {(gdf_yolo_pre['score'] > 0.10).sum()}")

## 4. Bbox area comparison — do SAM2 bboxes overlap more?

In [ ]:
gdf_sam2_pre["poly_area"]  = gdf_sam2_pre.geometry.area
gdf_yolo_pre["poly_area"]  = gdf_yolo_pre.geometry.area
gdf_sam2_post["poly_area"] = gdf_sam2_post.geometry.area
gdf_yolo_post["poly_area"] = gdf_yolo_post.geometry.area

print("SAM2 pre-NMS  — median poly area:", gdf_sam2_pre["poly_area"].median())
print("YOLO pre-NMS  — median poly area:", gdf_yolo_pre["poly_area"].median())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gdf_sam2_pre["poly_area"].clip(0, gdf_sam2_pre["poly_area"].quantile(0.99)),
        bins=50, alpha=0.6, label="SAM2 pre-NMS")
ax.hist(gdf_yolo_pre["poly_area"].clip(0, gdf_yolo_pre["poly_area"].quantile(0.99)),
        bins=50, alpha=0.6, label="YOLO pre-NMS")
ax.set_xlabel("polygon area (m²)"); ax.set_ylabel("count")
ax.set_title("Pre-NMS polygon area distributions")
ax.legend(); plt.tight_layout(); plt.savefig("diag_prenms_areas.png", dpi=150); plt.show()

## 5. Conclusion

- If SAM2 pre-NMS ≈ 14K → **Case 1**: masks failing before NMS → fix `process_SAM2`
- If SAM2 pre-NMS ≈ 44K → **Case 2**: NMS too aggressive on larger bboxes → fix NMS to use polygon IoU